# FULL-SYN — học lại 50 SKU ở task 6–7

[Plan](../../docs/project/task6-relearn-plan-2026-10-30.md) và [phân tích round 1](../../docs/project/full-syn-r1-analysis-2026-10-29.md), mục 9. Model task 5 của round 1 nhầm nhiều ở 50 SKU (32 SKU có recall hoặc precision dưới 85% trên val, cộng các SKU anh em hay bị nhầm với chúng và hai SKU chocolate thiếu mặt có nhãn). Thay vì train lại từ đầu, campaign này:

1. mở rộng model task 5 của round 1 (`seed_campaign`) lên 275 ô class, giữ nguyên output của 200 class đã học;
2. **task 6** và **task 7**, mỗi task học lại 25 SKU dưới class bí danh (alias), trên release ghép `rpc_synth_relearn_v1` có đủ các mặt của từng SKU;
3. chấm val như một model stage 5 của config cũ: dự đoán của alias được đổi về label gốc theo hai cách, `merge` (giữ cả ô cũ và alias, một vật đếm một lần) và `alias_only` (bỏ ô cũ của SKU đã học lại); chọn cách có cAcc-M2 cao hơn (bằng nhau thì `merge`).

| Cổng | Điều kiện để đi tiếp (`go`) |
|---|---|
| Sau task 6 | cAcc-M2 tăng ≥ 1,65 điểm so với model task 5 của round 1 **và** AP50 của 150 SKU không học lại giảm ≤ 1 điểm |
| Sau task 7 | cAcc-M2 tăng ≥ 3 điểm, cùng điều kiện AP50 |

Một cell báo `GateStop` nghĩa là cổng đã dừng campaign ở đó: không chạy task sau, không chấm test. Test2019 chỉ chấm một lần, sau `go` của task 7, khi người dùng tự đổi `CONFIRM_TEST = True`. Bị ngắt giữa chừng thì chạy lại chính file này: bước đã xong được bỏ qua, task dở chạy tiếp từ `last.ckpt`.

In [ ]:
import json, os, sys
from pathlib import Path
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').is_file())
sys.path.insert(0, str(REPO))
from autocheckout import relearn
from autocheckout.io import load_json
from autocheckout.taskcfg import TaskConfig
from autocheckout.training_workflow import resolve
SETTINGS_PATH = resolve(os.environ.get('AUTOCHECKOUT_TRAINING_SETTINGS', 'data/training/settings_full_syn_relearn.json'))
SETTINGS = load_json(SETTINGS_PATH)
PLAN = relearn.prepare(SETTINGS)
print('Settings:', SETTINGS_PATH)
print('Campaign:', PLAN['root'], '| round', PLAN['round'], '| dry run' if PLAN['dry_run'] else '| real run')
print('Synthetic config sha256:', PLAN['release_config_sha256'])
print('Relearn tasks:', PLAN['relearn_tasks'], '| train images per task:', PLAN['train_images'])
print('Class slots:', PLAN['n_classes'], '| SKUs relearned:', len(PLAN['relearned']))
print('Seed (round 1, task 5):', PLAN['seed_root'], PLAN['seed'])
print(json.dumps({k: PLAN[k] for k in ('resolution', 'batch_size', 'effective_batch', 'num_workers', 'tf32', 'epochs',
                                         'audit_images')}, indent=1))

In [ ]:
import platform, subprocess, torch
from autocheckout.runinfo import git_state
git = git_state(REPO)
print('git', git.get('commit'), '| dirty' if git.get('dirty') else '| clean')
print('Python', platform.python_version(), '| torch', torch.__version__, '| CUDA', torch.version.cuda)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv'],
                     capture_output=True, text=True).stdout if torch.cuda.is_available() else '')

## Mở rộng model task 5 của round 1

`pdp/expand.py` chép model task 5 của round 1 sang 275 ô class; output của 200 class đã học giữ nguyên. Link `runs/EXP-FULL-SYN-FSA` của round 1 (backbone sau FSA). Kiểm md5 của `task_5/task_final.pth` so với lúc chuẩn bị campaign. Ghi điểm của model round 1 trên val làm mốc cho hai cổng: cAcc-M2 và AP50 của các SKU không học lại.

In [ ]:
EXPAND = relearn.stage_expand(PLAN)
print('learned classes:', EXPAND['learned'], '| expanded checkpoint md5:', EXPAND['checkpoint_md5'])
print('seed scores on val:', json.dumps(EXPAND['seed_scores']))

## Task 6

Trước khi train: audit nhãn giả của model trước (chỉ ghi lại, không phải cổng). Sau khi train: chấm val theo hai cách đổi alias, so với task 5 của round 1, rồi qua cổng sau task 6.

In [ ]:
R1 = load_json(Path(PLAN['seed_root']) / 'stages/tasks.json')['tasks']['5']

def show(task):
    """Scores of a finished relearn task, also after a stop; nothing if the task did not finish (see the traceback)."""
    path = Path(PLAN['root']) / 'stages' / f'task{task}.json'
    if not path.is_file():
        return
    r = load_json(path)
    print(f"task {task}: mode {r['mode']} | gate {r['gate']['decision'].upper()} - {r['gate']['reason']}")
    print(f"{'':16s} {'cAcc-M2':>8s} {'M1 groups':>9s} {'AP50 all':>8s} {'AP50 other':>10s}")
    print(f"{'round 1 task 5':16s} {R1['cAcc_m2']:8.4f} {R1['cAcc_m1_groups']:9.4f} {R1['AP50']['mAP_A']:8.4f} "
          f"{r['seed']['AP50_other']:10.4f}")
    for mode, s in r['modes'].items():
        print(f"{mode:16s} {s['cAcc_m2']:8.4f} {s['cAcc_m1_groups']:9.4f} {s['AP50']['mAP_A']:8.4f} {s['AP50_other']:10.4f}")
    p = r['pseudo_labels']
    print('pseudo-label audit:', {k: p.get(k) for k in ('images', 'earlier_objects', 'recall')})
    print('top confusions:', r['modes'][r['mode']]['confusions_top20'][:8])

try:
    relearn.stage_task(PLAN, 6)
finally:
    show(6)

## Task 7

Như task 6, với cổng cuối (cAcc-M2 tăng ≥ 3 điểm). Thêm recall và precision của 50 SKU trên val, tính như lúc chọn SKU (policy M2, ghép vật và box theo IoU ≥ 0,5), và số SKU có recall hoặc precision dưới 85%: trước khi học lại là 32 trên 200 (`configs/relearn/selection.json`, mục `low`). Bảng này in cả khi cổng dừng, để biết SKU nào chưa học lại được.

In [ ]:
def show_accuracy(task):
    if not (Path(PLAN['root']) / 'stages' / f'task{task}.json').is_file():
        return
    acc = relearn.sku_accuracy(PLAN, task)
    names = TaskConfig.load(PLAN['seed_task_config']).label_names()
    worst = lambda label: min(acc[label]['recall'], acc[label]['precision'])
    print(f"{'label':>5s} {'name':24s} {'n val':>5s} {'recall':>7s} {'precision':>9s}")
    for label in sorted(PLAN['relearned'], key=worst):
        a = acc[label]
        print(f"{label:5d} {names[label][:24]:24s} {a['n_val']:5d} {a['recall']:7.3f} {a['precision']:9.3f}")
    low = [label for label in acc if acc[label]['n_val'] and worst(label) < 0.85]
    before = len(load_json(REPO / 'configs/relearn/selection.json')['low'])
    print(f"SKUs below 85% (recall or precision): {len(low)} of {len(acc)} (before relearning: {before}); "
          f"among the {len(PLAN['relearned'])} relearned: {len(set(low) & set(PLAN['relearned']))}")

try:
    relearn.stage_task(PLAN, 7)
finally:
    show(7)
    show_accuracy(7)

## Test2019 (một lần)

Chấm test bằng policy val của cách đổi alias đã chọn ở task 7, chỉ sau `go` của task 7. Đọc cổng task 7 ở trên, rồi tự đổi `CONFIRM_TEST = True` và chạy lại notebook (các bước đã xong được bỏ qua). Test đã chấm thì chỉ in lại, không chấm lại. Dry run chấm test của fixture mà không cần đổi.

In [ ]:
CONFIRM_TEST = False  # đổi thành True sau khi đọc cổng task 7
FINAL = relearn.stage_final_test(PLAN, confirm=CONFIRM_TEST or PLAN['dry_run'])
SEED_TEST = load_json(Path(PLAN['seed_root']) / 'stages/final_test.json')['test2019']
print('mode:', FINAL['mode'])
print('test AP50:', FINAL['test2019']['AP50'], '| round 1 task 5:', SEED_TEST['AP50']['5'])
for policy, stages in FINAL['test2019']['counting'].items():
    print(policy, json.dumps(stages), '| round 1 task 5:', json.dumps(SEED_TEST['counting'][policy]['5']))